## Polytope On-Demand-DT Country cut-out example notebook

This notebook shows how to use earthkit-data and earthkit-plots to pull destination-earth data from LUMI and plot it using earthkit-plots.

Before running the notebook you need to set up your credentials. See the main readme of this repository for different ways to do this or use the following cells to authenticate.

You will need to generate your credentials using the desp-authentication.py script.

This can be run as follows:

In [ ]:
%%capture cap
%run ../desp-authentication.py

This will generate a token that can then be used by earthkit and polytope.

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

# Requirements
To run this notebook install the following:
* pip install earthkit-data
* pip install earthkit-plots
* pip install earthkit-geo
* pip install earthkit-regrid  (Optional for spectral variables)
* pip install cf-units         (Optional for unit conversion in maps)

If you do not have eccodes installed please install eccodes using conda as it is a dependency of earthkit, or install earthkit via conda

* conda install eccodes -c conda-forge
* conda install earthkit-data -c conda-forge

In [ ]:
import earthkit.data
import earthkit.plots
from earthkit.geo import cartography

In [ ]:
# Defaults to making a live data request. Set to false to use the cached GRIB file instead.
import os

LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

In [ ]:
countries = ["Ireland"] # List of countries
shapes = cartography.country_polygons(countries, resolution=500e5)

# Remap longitude points to range [0, 360]
for shape in shapes:
    for point in shape:
        lon = point[1]
        if lon < 0:
            point[1] = lon + 360


In [ ]:
request = {
    "class": "d1",
    "dataset": "on-demand-extremes-dt",
    "stream": "oper",
    "type": "fc",
    "date": "20250926",
    "time": "0000",
    "levtype": "sfc",
    "expver": "0099",
    "param": "167",
    "step": "1/to/9",
    "georef": "gcgkrb",
    "feature": {
        "type": "polygon",
        "shape": shapes,
    },
}


In [ ]:
data_file = "data/on-demand-country-fe-example.covjson"
if LIVE_REQUEST:
    data = earthkit.data.from_source("polytope", "destination-earth", request, address="polytope.lumi.apps.dte.destination-earth.eu", stream=False)
    data.to_target("file", data_file)
else:
    data = earthkit.data.from_source("file", data_file)

In [ ]:
ds = data.to_xarray()
ds

In [ ]:
data_single_step = ds.sel(steps=1, number=0, datetimes='2025-09-26T00:00:00Z')

data_single_step

In [ ]:
import earthkit.plots
chart = earthkit.plots.Map(domain="Ireland")
chart.point_cloud(
    data_single_step['2t'],
    x="longitude",
    y="latitude",
)
chart.title(f"Results from on-demand-extremes-dt")
chart.coastlines()
chart.gridlines()
chart.legend()
chart.show()